In [5]:
import LowLevelFEM as FEM
using LowLevelFEM

gmsh.initialize();

In [6]:
gmsh.open("lapos-acel.geo")

mat1 = FEM.material("beam", E=2e5, ν=0.3)
mat2 = FEM.material("plate", E=2e7, ν=0.3)
problem = FEM.Problem([mat1, mat2], type=:Solid)

Info    : Reading 'lapos-acel.geo'...
Info    : Meshing 1D...                                                                                                             
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Line)
Info    : [ 10%] Meshing curve 3 (Line)
Info    : [ 20%] Meshing curve 4 (Line)
Info    : [ 20%] Meshing curve 5 (Line)
Info    : [ 30%] Meshing curve 6 (Line)
Info    : [ 30%] Meshing curve 7 (Line)
Info    : [ 30%] Meshing curve 8 (Line)
Info    : [ 40%] Meshing curve 9 (Line)
Info    : [ 40%] Meshing curve 10 (Line)
Info    : [ 50%] Meshing curve 11 (Line)
Info    : [ 50%] Meshing curve 12 (Line)
Info    : [ 60%] Meshing curve 13 (Line)
Info    : [ 60%] Meshing curve 14 (Line)
Info    : [ 60%] Meshing curve 15 (Line)
Info    : [ 70%] Meshing curve 16 (Line)
Info    : [ 70%] Meshing curve 17 (Line)
Info    : [ 80%] Meshing curve 18 (Line)
Info    : [ 80%] Meshing curve 19 (Line)
Info    : [ 80%] Meshing curve 20 (Line)
Info    : [ 90%] M

Problem("lapos-acel", :Solid, 3, 3, Material[Material("beam", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("plate", :Hooke, 2.0e7, 0.3, 1.1538461538461536e7, 7.692307692307692e6, 1.6666666666666664e7, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 25638, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f, false)

In [7]:
#gmsh.fltk.run()

In [8]:
function Fx(x, y, z)
    ξ = x - 1.5
    η = y - 15
    φ = atan(η, ξ)
    r = √(ξ^2 + η^2)
    return -r * sin(φ)
end

function Fy(x, y, z)
    ξ = x - 1.5
    η = y - 15
    φ = atan(η, ξ)
    r = √(ξ^2 + η^2)
    return r * cos(φ)
end

Fy (generic function with 1 method)

In [9]:
supp = FEM.displacementConstraint("supp", ux=0, uy=0, uz=0)
load = FEM.load("load", fx=Fx, fy=Fy);

In [10]:
K = FEM.stiffnessMatrix(problem)

sparse([1, 2, 3, 643, 644, 645, 646, 647, 648, 2791  …  76611, 76903, 76904, 76905, 76909, 76910, 76911, 76912, 76913, 76914], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  76914, 76914, 76914, 76914, 76914, 76914, 76914, 76914, 76914, 76914], [1.0075988519359654e8, 1.2290727135779308e6, -2.7617483401514854e7, 10683.760683760644, 8012.820512820658, 3205.128205128703, 1.45601262172974e6, -133297.69454277845, 334370.67218118, -3.415798756597688e7  …  -3.591533859931866e7, 31548.832016130167, 6.298319371672818e6, -4.034447608057897e6, 9.792885075727547e6, 1.0947817629188878e7, -1.9793059896478295e7, -6.285134871019693e6, -2.7168710144270703e7, 1.2492580837549776e8], 76914, 76914)

In [11]:
f = FEM.loadVector(problem, [load]);

In [12]:
FEM.applyBoundaryConditions!(problem, K, f, [supp]);

LoadError: MethodError: no method matching applyBoundaryConditions!(::Problem, ::SystemMatrix, ::VectorField, ::Vector{BoundaryCondition})
The function `applyBoundaryConditions!` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  applyBoundaryConditions!([91m::Union{ScalarField, TensorField, VectorField}[39m, [91m::Vector{BoundaryCondition}[39m)
[0m[90m   @[39m [35mLowLevelFEM[39m [90m~/Dokumentumok/GitHub/LowLevelFEM.jl/src/[39m[90m[4mlinear.jl:3522[24m[39m


In [ ]:
q = FEM.solveDisplacement(K, f)
S = FEM.solveStress(problem, q);

In [ ]:
u = FEM.showDoFResults(problem, q, "uvec", name="uvec", visible=false)
#ux = FEM.showDoFResults(problem, q, "ux", name="ux", visible=false)
#uy = FEM.showDoFResults(problem, q, "uy", name="uy", visible=false)

#s = FEM.showStressResults(problem, S, "s", name="σ", visible=true, smooth=false)
#ss = FEM.showStressResults(problem, S, "s", name="σ smooth", visible=false, smooth=true)
sx = FEM.showStressResults(problem, S, "sx", name="σx", visible=false, smooth=true)
sy = FEM.showStressResults(problem, S, "sy", name="σy", visible=false, smooth=true)
sz = FEM.showStressResults(problem, S, "sz", name="σz", visible=false, smooth=true)
sxy = FEM.showStressResults(problem, S, "sxy", name="τxy", visible=false, smooth=true)
syz = FEM.showStressResults(problem, S, "syz", name="τyz", visible=false, smooth=true)
szx = FEM.showStressResults(problem, S, "szx", name="τzx", visible=false, smooth=true)

Info    : Running Plugin(Smooth)...
Info    : Done running Plugin(Smooth)
Info    : Running Plugin(Smooth)...
Info    : Done running Plugin(Smooth)
Info    : Running Plugin(Smooth)...
Info    : Done running Plugin(Smooth)
Info    : Running Plugin(Smooth)...
Info    : Done running Plugin(Smooth)
Info    : Running Plugin(Smooth)...
Info    : Done running Plugin(Smooth)
Info    : Running Plugin(Smooth)...
Info    : Done running Plugin(Smooth)


7

In [ ]:
#FEM.plotOnPath(problem, "top", sy, 1000, name="σy", visible=false);

In [ ]:
gmsh.fltk.run()

-------------------------------------------------------
Version       : 4.13.1
License       : GNU General Public License
Build OS      : Linux64-sdk
Build date    : 19700101
Build host    : amdci7.julia.csail.mit.edu
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blossom Cairo DIntegration Dlopen DomHex Eigen[contrib] Fltk GMP Gmm[contrib] Hxt Jpeg Kbipack LinuxJoystick MathEx[contrib] Mesh Metis[contrib] Mmg Mpeg Netgen Nii2mesh ONELAB ONELABMetamodel OpenCASCADE OpenCASCADE-CAF OpenGL OpenMP OptHom Parser Plugins Png Post QuadMeshingTools QuadTri Solver TetGen/BR TinyXML2[contrib] Untangle Voro++[contrib] WinslowUntangler Zlib
FLTK version  : 1.3.8
OCC version   : 7.7.2
Packaged by   : root
Web site      : https://gmsh.info
Issue tracker : https://gitlab.onelab.info/gmsh/gmsh/issues
-------------------------------------------------------


XOpenIM() failed
XRequest.18: BadValue 0x0


In [ ]:
gmsh.finalize()